In [ ]:
"""
postprocess_exposure_wavg.ipynb
============================
Post-processing of GEE hazard-exposure & weighted-average CSVs.
 
Reads the per-hazard CSV files produced by the GEE workflow
(pop_w_exposure_gee.py, section 7) and generates:
 
  - One Excel + PNG map per EXPOSURE variant (per threshold: abs, gmean, cmean)
  - One Excel + PNG map per hazard for the WEIGHTED AVERAGE of hazard intensity
  - A JSON run log capturing all processing decisions and statistics
 
Each Excel file contains:
  admin_code | admin_name | indicator_value | {above|below}_mean | {above|below}_median
 
Each PNG is a choropleth map of the indicator over admin polygons, with
nodata regions rendered in grey.

Binarization direction
──────────────────────
  Exposure (% of population exposed): higher = more deprived → always "above".
  Weighted average: direction depends on the hazard's comparison field:
    comparison="gt" → higher values = worse → "above_mean/median"
    comparison="lt" → lower values = worse  → "below_mean/median"
 
  The CSV "comparison" column values: "gt" or "lt", from hazard metadata.

Nodata / zero handling
──────────────────────
  When zeros_as_nodata=True, indicator zeros are replaced with NaN directly
  on the indicator column (not on a copy). This means:
    - The binarization mean/median excludes these regions
    - The Excel file shows NaN for these regions
    - The PNG map renders these regions in grey (nodata layer)

Run log
───────
  A JSON file is written alongside the outputs capturing:
    - Run context (timestamp, country, admin level, paths, defaults)
    - Per-hazard resolved settings and how they were determined
    - Per-indicator binarization statistics and nodata counts
"""

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import json
 
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pycountry


In [ ]:
# =========================================================================
# 1. CONFIGURATION
# =========================================================================

# ── Folder structure ──
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_in_shapes = base_dir + "data_in/shapes/"

# Subfolder where the GEE per-hazard CSVs live (after downloading from Drive)
climate_dir = "d&a_climate/"

# Output folder
maps_exposure_folder = base_dir + "data_out/maps/exposure/"
Path(maps_exposure_folder).mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Country / admin selection ──
# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = wia_countries_exercise[0]
c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3
    if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# NOTE: select admin level and population year
admin_level = "2"
year = f"{date.today().year - 1}"

# ── Column names in the GEE CSVs ──
# These must match the id_name used in the GEE workflow and the shapefile.
# NOTE: this assumes harmonization across CODs, modify if other shapefiles used
pcode_col = f"adm{admin_level}_pcode" # "ADM1_CODE" #
name_col = f"adm{admin_level}_name" # "ADM1_NAME" #

# Suffix used in the GEE workflow (appears in CSV filenames)
# NOTE: modify accordingly if alternative shapefiles used
suffix = f"{c_iso3.lower()}_adm{admin_level}" # "vct_nso_adm1" #


In [ ]:
# ── Weighted-average strategy selection ──
# Default: use "total_pop" weighted average (denominator = all population).
# Override per hazard by adding an entry to hazard_settings below.
default_wavg_strategy = "total_pop"

# ── Zero-handling for binarization ──
# When GEE outputs 0 for an indicator (exposed pop or weighted avg), it may
# reflect genuine zero hazard OR missing data (nodata pixels → masked → sum=0).
# Including spurious zeros in the mean/median computation shifts the
# binarization threshold and can flip classifications.
#
# When zeros_as_nodata=True, zeros in the indicator column are replaced with
# NaN directly. This affects:
#   - The binarization (excluded from mean/median computation)
#   - The Excel output (NaN shown for these regions)
#   - The map (these regions rendered as grey nodata)
#
# Default: auto-detect from the "nodata_means_zero" column in the CSV.
#   nodata_means_zero=True  → zeros are physically real   → keep (zeros_as_nodata=False)
#   nodata_means_zero=False → zeros may be from nodata    → replace (zeros_as_nodata=True)
#
# Set to True/False explicitly to override the auto-detection for all hazards,
# or override per hazard in hazard_settings.
default_zeros_as_nodata = None  # None = auto-detect

# ── Per-hazard overrides ──
# Keys must match the "hazard_name" column in the CSVs.
# Only include hazards that need non-default behaviour.
hazard_settings = {
    # Example: force intersection-weighted average for drought indices
    # "drought_spei_copernicus_2014-2024": {
    #     "wavg_strategy": "intersection",
    #     "zeros_as_nodata": True,
    # },
    # Example: keep zeros as real for flood exposure
    # "river_flood_100yr_jrc_2024": {
    #     "zeros_as_nodata": False,
    # },
    # keep zeros as real data for HW freq. (number)
    "heatwave_frequency_ecmwf_2014-2024": {
        "zeros_as_nodata": False,
    },
    # keep zeros as real data for HW dur. (number of days)
    "heatwave_duration_ecmwf_2014-2024": {
        "zeros_as_nodata": False,
    },
    # keep zeros as real data for HW sev. (ºC)
    "heatwave_severity_ecmwf_2014-2024": {
        "zeros_as_nodata": False,
    },
    # keep zeros as real data for Extreme Heat (number of days)
    "extreme_heat_ecmwf_2014-2024": {
        "zeros_as_nodata": False,
    },
    # keep zeros as real data for LS (number of events)
    "LS_RF_GFDRR_1980-2018": {
        "zeros_as_nodata": False,
    },
    # use intersection-weighted average for FAO agricultural drought
    "agricultural_drought_fao_1984-2023": {
        "wavg_strategy": "intersection",
    },
    # keep zeros as real data for SPEI drought (exposed population)
    "drought_spei_copernicus_1940-2024": {
        "wavg_strategy": "intersection",
        "zeros_as_nodata": False,
    },
    # keep zeros as real data for SPI drought (exposed population)
    "drought_spi_copernicus_1940-2024": {
        "wavg_strategy": "intersection",
        "zeros_as_nodata": False,
    },
}


In [ ]:
# =========================================================================
# 2. LOAD SHAPEFILE
# =========================================================================

# geojson output file name
# NOTE: modify accordingly if alternative shapefiles used
geo_filename = f"geojson_{c_iso3}_adm{admin_level}" # f"geojson_{c_iso3}_NSO_adm{admin_level}" #
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(data_in_shapes + geojson_file[0]).to_crs("EPSG:4326")
geojson_dict = json.loads(gdf.to_json())


In [ ]:
# =========================================================================
# 3. LOAD GEE CSVs
# =========================================================================

csv_dir = Path(data_in_folder + climate_dir)
exposure_csvs = sorted([
    f for f in csv_dir.iterdir() if f.name.endswith(f"_{suffix}.csv")
])

print(f"Found {len(exposure_csvs)} hazard CSV files for {suffix}")


In [ ]:
# =========================================================================
# 4. HELPER FUNCTIONS
# =========================================================================

# ── Nodata grey colour ──
NODATA_GREY = "Gray" # "lightgrey" #

# ── Colour scales ──
# Exposure: always YlOrRd (higher % exposed = worse = red)
# Weighted average: depends on comparison direction
CSCALE_GT = "YlOrRd"    # higher = worse → red at top
CSCALE_LT = "YlOrRd_r"  # lower = worse  → red at bottom (reversed)


In [ ]:
def resolve_setting(hazard_name, key, default, nodata_means_zero=None):
    """
    Look up a setting: per-hazard override → explicit default → auto-detect.

    Returns (value, source) where source is one of:
      "override"     — from hazard_settings dict
      "default"      — from the explicit default_* variable
      "auto_detect"  — inferred from nodata_means_zero in the CSV
      "fallback"     — none of the above applied, used hard-coded False
    """
    per_hazard = hazard_settings.get(hazard_name, {})
    if key in per_hazard:
        return per_hazard[key], "override"
    if default is not None:
        return default, "default"
    # Auto-detect from nodata_means_zero (only for zeros_as_nodata)
    if key == "zeros_as_nodata" and nodata_means_zero is not None:
        return (not nodata_means_zero), "auto_detect"
    return False, "fallback"


In [ ]:
def compute_binarization(
    df, indicator_col, zeros_as_nodata, comparison, pcode_col, name_col
):
    """
    Given a DataFrame with an indicator column, compute binarization columns
    and return the slim output DataFrame.
    The indicator is binarized to identify the most deprived subnational areas.

    For exposure (always pass comparison="gt"):
      above_mean = 1 where indicator >= mean
    For weighted average:
      comparison="gt" → higher = worse → above_mean / above_median
      comparison="lt" → lower  = worse → below_mean / below_median
 
    NaN indicators get NaN in the binarization columns.

    Parameters
    ----------
    df : DataFrame with at least pcode_col, name_col, and indicator_col
    indicator_col : str — name of the column holding the indicator values
    zeros_as_nodata : bool — if True, replace indicator=0 as NaN
    before computing mean/median

    Modifying the indicator ensures that:
      - NaN propagates to the Excel output
      - NaN regions show as grey in the map
      - Mean/median computation for binarization excludes them

    Returns
    -------
    out_df : DataFrame with columns: pcode, name, indicator, {dir}_mean, {dir}_median
    stats : dict with binarization statistics for the log
    """
    out = df[[pcode_col, name_col, indicator_col]].copy()

    if zeros_as_nodata:
        # Replace exact zeros with NaN in-place on the indicator series
        # These rows will get NaN in binarization and indicator columns.
        # Returns later the count of zeros replaced.
        n_replaced = int((out[indicator_col] == 0).sum())
        out[indicator_col] = out[indicator_col].replace(0, pd.NA)
    else:
        n_replaced = 0

    # Series used for binarization statistics
    vals = out[indicator_col]
    stat_mean = vals.mean()
    stat_median = vals.median()
    n_total = len(vals)
    n_nodata = int(vals.isna().sum())
    n_data = n_total - n_nodata

    # Binarize accordingly
    if comparison == "gt":
        prefix = "above"
        out[f"{prefix}_mean"] = (vals >= stat_mean).astype(int)
        out[f"{prefix}_median"] = (vals >= stat_median).astype(int)
    else:
        prefix = "below"
        out[f"{prefix}_mean"] = (vals <= stat_mean).astype(int)
        out[f"{prefix}_median"] = (vals <= stat_median).astype(int)

    # Ensure if the indicator is NaN → binarization is also NaN
    nodata_mask = vals.isna()
    out.loc[nodata_mask, f"{prefix}_mean"] = pd.NA
    out.loc[nodata_mask, f"{prefix}_median"] = pd.NA

    n_deprived_mean = int(out[f"{prefix}_mean"].sum())
    n_deprived_median = int(out[f"{prefix}_median"].sum())

    stats = {
        "direction": prefix,
        "comparison": comparison,
        "binarization_mean": round(stat_mean, 6) if pd.notna(stat_mean) else None,
        "binarization_median": round(stat_median, 6) if pd.notna(stat_median) else None,
        "zeros_replaced_with_nan": n_replaced,
        "n_regions_total": n_total,
        "n_regions_with_data": n_data,
        "n_regions_nodata": n_nodata,
        "n_deprived_by_mean": n_deprived_mean,
        "n_deprived_by_median": n_deprived_median,
    }

    return out, stats


In [ ]:
def save_map_png(
    df,
    geojson_dict,
    pcode_col,
    indicator_col,
    title_text,
    out_path,
    color_scale="YlOrRd",
):
    """
    Render a choropleth and write to PNG.
 
    Regions with NaN in the indicator are rendered as a separate grey layer
    underneath the data choropleth. This makes nodata regions visible
    rather than simply omitting them from the map.
 
    Approach: two go.Choropleth traces sharing the same GeoJSON.
      Trace 1 (bottom): nodata features — uniform grey, no colorbar.
      Trace 2 (top):    data features   — coloured by indicator value.
 
    We drop down from px.choropleth to go.Figure + go.Choropleth so that
    we can stack two traces. The visual result should be the same as the original
    px.choropleth call, with the addition of the grey nodata layer.
    """

    has_nodata = df[indicator_col].isna().any()
    data_df = df.dropna(subset=[indicator_col])

    fig = go.Figure()

    # ── Trace 1: nodata regions in grey ──
    if has_nodata:
        nodata_df = df[df[indicator_col].isna()]
        fig.add_trace(go.Choropleth(
            geojson=geojson_dict,
            featureidkey=f"properties.{pcode_col}",
            locations=nodata_df[pcode_col],
            z=[0] * len(nodata_df),
            colorscale=[[0, NODATA_GREY], [1, NODATA_GREY]],
            showscale=False,
            hoverinfo="location+text",
            text=["No data"] * len(nodata_df),
            marker_line_color="Gainsboro",
            marker_line_width=0.5,
        ))

    # ── Trace 2: data regions with colour scale ──
    fig.add_trace(go.Choropleth(
        geojson=geojson_dict,
        featureidkey=f"properties.{pcode_col}",
        locations=data_df[pcode_col],
        z=data_df[indicator_col],
        colorscale=color_scale,
        colorbar=dict(len=0.65),
        marker_line_color="Gainsboro",
        marker_line_width=0.5,
    ))

    fig.update_geos(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
    )
    fig.update_layout(
        title=dict(text=title_text, x=0.5, y=0.97),
        margin={"r": 0, "t": 0, "l": 0, "b": 0},
        width=1200,
        height=900,
    )

    fig.write_image(out_path, format="png", scale=2)


In [ ]:
# =========================================================================
# 5. MAIN PROCESSING LOOP
# =========================================================================

# Disable automatic rendering
pio.renderers.default = None

sheet_name = f"{c_iso3}_adm{admin_level}_{year}"

# ── Initialise run log ──
run_log = {
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "country": country,
    "iso3": c_iso3,
    "admin_level": admin_level,
    "year": year,
    "suffix": suffix,
    "shapefile": str(geojson_file[0]),
    "n_admin_regions": len(gdf),
    "n_hazard_csvs": len(exposure_csvs),
    "defaults": {
        "wavg_strategy": default_wavg_strategy,
        "zeros_as_nodata": (
            default_zeros_as_nodata if default_zeros_as_nodata is not None
            else "auto_detect"
        ),
    },
    "hazard_settings_overrides": {
        k: v for k, v in hazard_settings.items() if v
    },
    "hazards": [],
}

for csv_path in exposure_csvs:
    df = pd.read_csv(csv_path)

    df[pcode_col] = df[pcode_col].astype(str)

    # ── Join admin names from shapefile ──
    df = df.merge(gdf[[pcode_col, name_col]], on=pcode_col, how="left")

    hazard_name = df["hazard_name"].iloc[0]
    nodata_flag = bool(df["nodata_means_zero"].iloc[0])
    comparison = str(df["comparison"].iloc[0])  # "gt" or "lt"

    # Resolve per-hazard settings (with source tracking)
    wavg_strategy, wavg_source = resolve_setting(
        hazard_name, "wavg_strategy", default_wavg_strategy
    )
    zeros_as_nodata, zeros_source = resolve_setting(
        hazard_name, "zeros_as_nodata", default_zeros_as_nodata,
        nodata_means_zero=nodata_flag,
    )

    print(
        f"── {hazard_name}  "
        f"[cmp={comparison}, wavg={wavg_strategy} ({wavg_source}), "
        f"zeros_as_nodata={zeros_as_nodata} ({zeros_source})]"
    )

    # ── Start hazard log entry ──
    hazard_log = {
        "hazard_name": hazard_name,
        "source_csv": csv_path.name,
        "comparison": comparison,
        "nodata_means_zero": nodata_flag,
        "settings": {
            "wavg_strategy": {
                "value": wavg_strategy,
                "source": wavg_source,
            },
            "zeros_as_nodata": {
                "value": zeros_as_nodata,
                "source": zeros_source,
            },
        },
        "indicators": [],
    }

    # ──────────────────────────────────────────────────────────────────────
    # A. EXPOSURE outputs (one per threshold variant)
    # ──────────────────────────────────────────────────────────────────────
    # Detect exposure columns dynamically (pop_exposed_abs, _gmean, _cmean)
    exp_cols = [c for c in df.columns if c.startswith("pop_exposed_")]

    for exp_col in exp_cols:
        label = exp_col.replace("pop_exposed_", "")  # e.g. "abs", "gmean", "cmean"

        # Compute exposure percentage
        indicator_col = f"exposed_pop_pct_{label}"
        df[indicator_col] = (df[exp_col] / df["total_population"] * 100).round(2)

        # Binarize — exposure is always "above" (higher % exposed = worse)
        out_df, bin_stats = compute_binarization(
            df, indicator_col, zeros_as_nodata, "gt", pcode_col, name_col
        )

        # mask admins without pop
        mask_no_pop = df["total_population"].isnull()
        # track number of admins without pop
        n_no_pop = mask_no_pop.sum()
        # add to bin_stats
        bin_stats["n_regions_without_pop"] = int(n_no_pop)

        # Where total_population is 0 → NaN from division; fill with 0
        # (no people = no exposure, regardless of hazard)
        out_df.loc[
            mask_no_pop,
            [indicator_col, "above_mean", "above_median"],
        ] = 0

        # File naming
        file_stem = f"{suffix.upper().replace('ADM', 'adm')}_{year}_{hazard_name}_exp_{label}"
        xlsx_path = maps_exposure_folder + file_stem + ".xlsx"
        png_path = maps_exposure_folder + file_stem + ".png"

        # add nodata fillna
        out_df.fillna("no-data").sort_values(
            by=[pcode_col]
        ).to_excel(
            xlsx_path,
            index=False,
            sheet_name=sheet_name,
        )

        # Exposure maps: always YlOrRd (higher % = worse = red)
        title = (
            f"Pop. exposed [%] — {hazard_name} ({label}) — "
            f"{suffix.upper().replace('ADM', 'adm')}_{year}"
        )
        save_map_png(
            out_df,
            geojson_dict,
            pcode_col,
            indicator_col,
            title,
            png_path,
            color_scale=CSCALE_GT,
        )
        print(f"  ✅  exp_{label}: {xlsx_path}")

        # ── Log this indicator ──
        hazard_log["indicators"].append({
            "type": "exposure",
            "label": label,
            "indicator_col": indicator_col,
            "color_scale": CSCALE_GT,
            "binarization": bin_stats,
            "outputs": {
                "xlsx": xlsx_path,
                "png": png_path,
            },
        })

    # ──────────────────────────────────────────────────────────────────────
    # B. WEIGHTED AVERAGE output (one per hazard layer)
    # ──────────────────────────────────────────────────────────────────────
    wavg_col = f"wavg_{wavg_strategy}"
    if wavg_col not in df.columns:
        print(f"  ⚠️  Column {wavg_col} not found — skipping weighted average")
        hazard_log["indicators"].append({
            "type": "wavg",
            "skipped": True,
            "reason": f"Column {wavg_col} not found in CSV",
        })
        run_log["hazards"].append(hazard_log)
        continue

    indicator_col = "hazard_wavg"
    df[indicator_col] = df[wavg_col].round(6)

    # Binarize — direction depends on hazard comparison
    out_wavg, bin_stats_wavg = compute_binarization(
        df, indicator_col, zeros_as_nodata, comparison, pcode_col, name_col
    )

    file_stem = f"{suffix.upper().replace('ADM', 'adm')}_{year}_{hazard_name}_wavg"
    xlsx_path = maps_exposure_folder + file_stem + ".xlsx"
    png_path = maps_exposure_folder + file_stem + ".png"

    # add nodata fillna
    out_wavg.fillna("no-data").sort_values(
        by=[pcode_col]
    ).to_excel(
        xlsx_path,
        index=False,
        sheet_name=sheet_name,
    )
    
    # Weighted-average maps: colour scale follows comparison direction
    wavg_cscale = CSCALE_GT if comparison == "gt" else CSCALE_LT
    title = (
        f"Pop-weighted avg intensity — {hazard_name} — "
        f"{suffix.upper().replace('ADM', 'adm')}_{year}"
    )
    save_map_png(
        out_wavg,
        geojson_dict,
        pcode_col,
        indicator_col,
        title,
        png_path,
        color_scale=wavg_cscale,
    )
    print(f"  ✅  wavg:   {xlsx_path}")

    # ── Log this indicator ──
    hazard_log["indicators"].append({
        "type": "wavg",
        "wavg_strategy": wavg_strategy,
        "indicator_col": indicator_col,
        "color_scale": wavg_cscale,
        "binarization": bin_stats_wavg,
        "outputs": {
            "xlsx": xlsx_path,
            "png": png_path,
        },
    })

    run_log["hazards"].append(hazard_log)


In [ ]:
# =========================================================================
# 6. WRITE RUN LOG
# =========================================================================

log_path = maps_exposure_folder + f"postprocess_log_{suffix}.json"
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=4)

print(f"\n✅  Run log saved: {log_path}")
print("\n✅  Post-processing complete.")
